# dh_tool.dataframe 테스트

이 노트북은 dh_tool.dataframe 패키지의 주요 기능들을 테스트합니다.

In [2]:
import pandas as pd
import numpy as np
import sys
from dh_tool.dataframe import DataFrame, Sheets
from dh_tool.dataframe_new.core.dataframe_core import DataFrameCore
import tempfile
import os
import matplotlib.pyplot as plt
import seaborn as sns

## 1. 테스트 데이터 준비

In [3]:
# 샘플 데이터프레임 생성
data = {
    'id': range(1, 6),
    'name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'age': [25, 30, np.nan, 35, 28],
    'score': [85.5, 90.0, 88.5, np.nan, 95.0],
    'tags': [['python', 'sql'], ['java'], ['python'], ['c++', 'java'], ['python', 'r']]
}
sample_df = pd.DataFrame(data)
df_handler = DataFrameCore(sample_df)

print("샘플 데이터프레임:")
display(sample_df)

샘플 데이터프레임:


,id,name,age,score,tags
0,1,Alice,25.0,85.5,"[python, sql]"
1,2,Bob,30.0,90.0,[java]
2,3,Charlie,NaN,88.5,[python]
3,4,David,35.0,NaN,"[c++, java]"
4,5,Eve,28.0,95.0,"[python, r]"


## 2. DataFrame 초기화 테스트

In [4]:
# DataFrame 초기화 확인
print("DataFrame 타입 확인:", isinstance(df_handler.df, pd.DataFrame))
print("데이터 길이 일치 확인:", len(df_handler.df) == len(sample_df))
print("컬럼 일치 확인:", all(df_handler.df.columns == sample_df.columns))

DataFrame 타입 확인: True
데이터 길이 일치 확인: True
컬럼 일치 확인: True


## 3. 데이터 선택 및 필터링 테스트

In [5]:
df_handler

In [6]:
# 단일 조건 테스트
result1 = df_handler.select_rows(include={'age': ('>', 28)})
print("28세 초과 데이터:")
display(result1)

# 복합 조건 테스트
result2 = df_handler.select_rows(
    include={'age': ('>', 25)},
    exclude={'name': ('contains', 'Bob')}
)
print("\n25세 초과이면서 이름에 'Bob'이 포함되지 않은 데이터:")
display(result2)

28세 초과 데이터:


,id,name,age,score,tags
1,2,Bob,30.0,90.0,[java]
3,4,David,35.0,NaN,"[c++, java]"



25세 초과이면서 이름에 'Bob'이 포함되지 않은 데이터:


,id,name,age,score,tags
3,4,David,35.0,NaN,"[c++, java]"
4,5,Eve,28.0,95.0,"[python, r]"


In [8]:
df_handler.df

,id,name,age,score,tags
0,1,Alice,25.0,85.5,"[python, sql]"
1,2,Bob,30.0,90.0,[java]
2,3,Charlie,NaN,88.5,[python]
3,4,David,35.0,NaN,"[c++, java]"
4,5,Eve,28.0,95.0,"[python, r]"


## 4. 결측값 처리 테스트

In [9]:
# 결측값 채우기 테스트
result = df_handler.fill_missing(strategy='mean', columns=['age', 'score'])
print("결측값 처리 결과:")
display(result)

print("\n결측값 존재 여부:")
print("age 열:", result['age'].isna().any())
print("score 열:", result['score'].isna().any())

AttributeError: 'DataFrameCore' object has no attribute 'fill_missing'

## 5. 정규화 테스트

In [ ]:
# 정규화 테스트
result = df_handler.normalize(columns=['age', 'score'])
print("정규화 결과:")
display(result)

print("\n정규화 범위 확인:")
print("age 범위:", result['age'].min(), "-", result['age'].max())
print("score 범위:", result['score'].min(), "-", result['score'].max())

## 6. Excel 처리 테스트

In [10]:
# Excel 저장 및 로드 테스트
with tempfile.NamedTemporaryFile(suffix='.xlsx', delete=False) as tmp:
    try:
        df_handler.save(tmp.name)
        print(f"파일 저장 성공: {os.path.exists(tmp.name)}")
        print(f"파일 크기: {os.path.getsize(tmp.name)} bytes")
    finally:
        os.unlink(tmp.name)

AttributeError: 'DataFrameCore' object has no attribute 'save'

## 7. Sheets 기능 테스트

In [ ]:
# Sheets 클래스 테스트
data1 = pd.DataFrame({'A': [1, 2, 3]})
data2 = pd.DataFrame({'B': [4, 5, 6]})

sheets = Sheets(data1)

# 새 시트 생성
sheets.create_sheet(data2, "Sheet2")
print("시트 목록:", sheets.sheet_names)

# 시트 선택
sheets.select_sheet("Sheet2")
print("\n선택된 시트 데이터:")
display(sheets.df)

# 시트 제거
sheets.remove_sheet("Sheet2")
print("\n시트 제거 후 목록:", sheets.sheet_names)

## 8. 시각화 테스트

In [ ]:
# 히스토그램 테스트
df_handler.plot_histogram('age')
plt.title('Age Distribution')
plt.show()

# 산점도 테스트
df_handler.plot_scatter('age', 'score')
plt.title('Age vs Score')
plt.show()

## 9. 에러 처리 테스트

In [ ]:
df_handler

In [ ]:
# 존재하지 않는 열 접근 테스트
try:
    df_handler.select_rows(include={'non_existent_column': ('>', 10)})
except KeyError as e:
    print("예상된 KeyError 발생:", e)

# 잘못된 연산자 테스트
try:
    df_handler.select_rows(include={'age': ('invalid_operator', 10)})
except ValueError as e:
    print("예상된 ValueError 발생:", e)

## 10. DataFrameHandler 메서드 테스트

In [ ]:
# 그룹화 및 집계 테스트
result = df_handler.group_and_aggregate('name', score='mean')
print("그룹화 결과:")
display(result)

# 중복 제거 테스트
result = df_handler.df_handler.drop_duplicates()
print("\n중복 제거 결과:")
display(result)